<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/K21_Konsult_51.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — Interpretive Jyotiṣa

**Khona21** is an experimental AI-assisted Jyotiṣa interpretation platform, developed as a companion to **Parashar21**, which performs the astronomical calculations underlying traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform combines precomputed Rashi and Navamsha charts, Vimshottari Dasha schedules and planetary transits (Gochar) with a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on hard-coded interpretive rules, Khona21 employs **source-grounded, multipass LLM reasoning** to examine planetary configurations, yogas, cancellations, exceptions and conflicting interpretations. An initial assessment establishes the horoscope's principal characteristics, while subsequent consultations investigate specific questions using the original reference texts and relevant astrological evidence.

This notebook, **k21_KhonaPlus**, implements the consultation workflow, integrating deterministic calculations from Parashar21 with the interpretive capabilities of modern Large Language Models.

**Khona21 explores the convergence of traditional Jyotiṣa knowledge and modern artificial intelligence, combining computational precision with source-grounded reasoning to produce systematic, transparent and auditable astrological interpretations.**

In [1]:
print("\033[1m"+'জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।'+"\033[0m")
!TZ='Asia/Kolkata' date

জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।
Sun Nov 15 10:28:11 AM IST 2026


#Setup, Installs, Imports

In [2]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown
from datetime import datetime
import pytz


!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
!wget -O khona.png -q https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/khana-sign-0.png

from k21core import *
from k21fileops import *

In [14]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
import k21core
import k21fileops
import importlib

importlib.reload(k21core)
importlib.reload(k21fileops)
from k21core import *
from k21fileops import *

In [ ]:
# ------------------------------------------------
# Use Claude
# ------------------------------------------------
#!pip install -q anthropic

#provider = "claude"
#model = "claude-haiku-5-5"
#model = "claude-sonnet-5-5"


In [4]:
# ------------------------------------------------
# Use OpenAI
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
#model = "gpt-6-luna"
model = "gpt-6.1-sol"
#model = "gpt-6-astra"

In [5]:
# ------------------------------------------------
# Set Native ID
# ------------------------------------------------
Native_id = "TopGun"

###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [6]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


In [7]:
# ------------------------------------------------
# Locate necessary file and set up paths to the same
# ------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
#ls /content/drive/MyDrive/Prithwis/Parashar21/K21

K21_HOME = "/content/drive/MyDrive/Prithwis/Parashar21/K21"

# ------------------------------------------------

BRAHA   = f"{K21_HOME}/Corpus/BR_Curated_22.txt"
HARIHAR = f"{K21_HOME}/Corpus/HM_Clean_07.txt"
FIRSTPASS = f"{K21_HOME}/FirstPass/"
#RAMAN = f"{K21_HOME}/Corpus/Raman_08.md"
#YOGS = f"{K21_HOME}/Corpus/300_Astrology_Yogas.md"



Mounted at /content/drive


## Load Files

In [8]:
#
# Following files from Parashar21 need to be uploaded here
#

import zipfile
from google.colab import files

# Upload ZIP file
uploaded = files.upload()
zipName = next(iter(uploaded))

# Extract all files into current directory
with zipfile.ZipFile(zipName, 'r') as zf:
    zf.extractall()

print("Extracted:", zipName)

CHART = "/content/Chart.txt"
NAVAMSA = "/content/Navamsa_Chart.txt"
DASHA = "/content/Dasha.txt"
GOCHAR = "/content/Gochar_Chart.txt"
GOCHAR_EVENTS = "/content/Transit.txt"

Saving Kompute_261115_092624.zip to Kompute_261115_092624.zip
Extracted: Kompute_261115_092624.zip


# LLM Access




In [9]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
ROLE = f"{K21_HOME}/Roles/khona261029A.txt"
khona21 = readText(ROLE)

/content/drive/MyDrive/Prithwis/Parashar21/K21/Roles/khona261029A.txt | 6,774 characters


In [10]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)

BaseCONTEXT = getContext(
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA
)


In [11]:
len(BaseCONTEXT)

944654

## the FirstPass

In [12]:
# ------------------------------------------------
# FIRST PASS
#
# ------------------------------------------------

FirstPass_Instruction = """
FIRST PASS — FOUNDATIONAL ASSESSMENT

Prepare a concise, high-level assessment of this horoscope.

Present an overall picture of the native's principal
strengths, weaknesses and significant planetary configurations.

Identify the most important favourable and unfavourable yogas,
sign exchanges, cancellations and mitigating factors,
to the extent that they are evident from the supplied data
and reference texts.

Begin with a simple-language summary, followed by a brief
astrological explanation with relevant source references.

This is an initial orientation, not an exhaustive audit.
Focus on the dominant features of the horoscope rather than
attempting to catalogue every planetary condition or yoga.

Distinguish chart facts from interpretations, and acknowledge
important contradictions or uncertainties.

Do not invent configurations or use rules unsupported by
the supplied reference texts.

Do not make predictions about specific future events.
"""

In [15]:
# ------------------------------------------------
# FirstPass: Load from cache or generate and save
# ------------------------------------------------

from pathlib import Path

cachedFiles = sorted(Path(FIRSTPASS).glob(f"{Native_id}_*.md"))

if cachedFiles:
    # Existing FirstPass
    MD_FILE = str(cachedFiles[-1])
    FirstPass = Path(MD_FILE).read_text(encoding="utf-8")
    print(f"FirstPass loaded: {MD_FILE}")

else:
    # Generate new FirstPass
    QUERY = buildQuery(BaseCONTEXT, FirstPass_Instruction)

    FirstPass = callLLM(
        client=client,
        role=khona21,
        query=QUERY,
        model=model,
        provider=provider,
        effort="medium"
    )

    # Save locally, then copy to Google Drive
    MD_FILE = saveAnswer(model, FirstPass)
    _ = saveFirstPass(MD_FILE, Native_id, FIRSTPASS)

    print(f"FirstPass generated and saved: {MD_FILE}")





OPENAI | gpt-6.1-sol | Effort: medium | In: 222,653 | Cached: 222,650 | Out: 2,781 | Reasoning: 464 | Time: 74.8s
Saved: /content/K21_261115_104539_gpt-6.1-sol.md
FirstPass saved: /content/drive/MyDrive/Prithwis/Parashar21/K21/FirstPass/TopGun_261115_gpt-6.1-sol.md
FirstPass generated and saved: /content/K21_261115_104539_gpt-6.1-sol.md


In [16]:
# ------------------------------------------------
# Display FirstPass (both cases)
# ------------------------------------------------
display(Markdown(FirstPass))

## 1. Simple-language summary

This horoscope suggests **ability, initiative and potential for professional achievement, but with substantial obstacles to ease, stability and personal contentment**. Its strongest constructive feature is Mars, the lord of both talent and career, placed prominently and aspecting its own career house. Intellectual, technical, commercial and communication-related abilities are also indicated.

The main vulnerabilities concern **domestic peace, partnerships, financial consistency and emotional well-being**. Several important planets have strengths that are counterbalanced by serious weaknesses. Most notably, Saturn occupies its own sign but is extremely close to its beginning; Jupiter is almost exactly debilitated, although cancellation conditions apply.

**Overall:** this is neither an unqualified prosperity chart nor an entirely adverse one. Its favourable combinations indicate resources for achievement, while its afflictions suggest that success and satisfaction need not arrive together.

These are interpretations within the supplied texts, not established facts about the native’s life.

## 2. Principal favourable configurations

### Mars supports career, intelligence and purposeful effort

**Chart facts:** Mars rules the 5th and 10th houses, occupies the angular 4th house, and aspects its own 10th house. It conjoins Mercury and receives Saturn’s aspect.

**Interpretation:** Mars is the principal constructive planet for Cancer ascendant. Its placement connects intelligence and creative ability with career, education and property. Its aspect to its own career house supports professional effectiveness, although the afflictions discussed below prevent an uncomplicated reading.

- **Braha, “Raja Yogakarakas”:** Mars alone is a raja-yogakaraka for Cancer ascendant.
- **Harihar, Chapter X, “Karkata Lagna”:** Mars requires a relationship with another favourable planet to deliver its fullest results. Here, its aspect to Jupiter, the 9th lord, supplies a favourable connection—but Jupiter’s debility limits the support.
- **Braha, “Planetary Aspects”:** an aspect to a planet’s own house can provide additional support.

### Mercury–Venus sign exchange is important

**Chart facts:** Mercury occupies **Libra, Venus’s sign**, while Venus occupies **Virgo, Mercury’s sign**. This is an established **Kshetra-binimay / Parivartana**, exchanging the **3rd and 4th houses**. Mercury additionally rules the 12th; Venus additionally rules the 11th.

**Interpretation:** The exchange strengthens the relationship between communication, self-effort and practical skills on one side, and education, home and property on the other. It also links gains and expenditure to those matters.

It is **not an unqualified auspicious exchange**: Venus is debilitated, Mercury is retrograde and afflicted, and the 12th lord participates.

**Sources:** Braha, “Exchange of Signs” and “Important Conditions Not Mentioned in Shad Bala”; Harihar, Chapter V, “Dhana Yoga,” on wealth-producing relationships between relevant house lords, qualified by their condition.

### Navamsha provides meaningful reinforcement

**Chart facts:** The Moon occupies its own sign Cancer in Navamsha. Saturn remains in Capricorn, its own sign, making it **vargottama**. The Sun also remains in Virgo and is vargottama.

**Interpretation:** These are genuine supporting factors, especially for the Moon’s underlying capacity and Saturn’s endurance. Neither Jupiter nor Venus remains debilitated in Navamsha. Nevertheless, divisional strength does not erase natal affliction.

**Source:** Braha, “The Navamsa” and “Sthanna Bala.”

## 3. Principal weaknesses and planetary balance

The following compact review distinguishes **strength from favourable functioning**.

| Planet | Dominant assessment |
|---|---|
| **Sun — 2nd lord** | In the 3rd, an upachaya favourable to the Sun; conjoins Venus and receives Jupiter’s aspect. Supports initiative and communication-related earning. At **28°49′ Virgo**, however, it is late in the sign and weakened under Braha’s planetary-zone rule. Virgo vargottama provides compensation. |
| **Moon — ascendant lord** | In the 6th, placing personal well-being under a difficult house influence. It is **waxing**, but not near full, and occupies its own Navamsha sign. These mitigate rather than eliminate the 6th-house weakness. No supplied full planetary aspect reaches it. |
| **Mars — 5th/10th lord** | Functionally favourable and angular, with an aspect to its own 10th. However, Mercury’s conjunction and Saturn’s mutual aspect complicate its expression, particularly in domestic matters. Aquarius Navamsha is favourable under Braha’s specific Mars-in-Aquarius observation. |
| **Mercury — 3rd/12th lord** | In friendly Libra, angular and exchanging signs with Venus. But it is retrograde, conjoined Mars and aspected by Saturn. Thus intellectual capacity is present alongside substantial affliction; its 12th lordship also burdens the 4th house. |
| **Jupiter — 6th/9th lord** | At **4°47′ Capricorn**, almost exactly its deepest debility at **5°**. Angular placement, conjunction with own-sign Saturn and Mars’s aspect provide significant qualifications, including cancellation conditions. It remains a mixed influence, not an unrestricted benefic. |
| **Venus — 4th/11th lord** | Debilitated in Virgo, though far from its deepest debility at **27°**. Its exchange with Mercury, Jupiter’s aspect and Aquarius Navamsha offer mitigation. Home, contentment and gains remain mixed rather than uniformly denied. |
| **Saturn — 7th/8th lord** | Own-sign, angular, directionally strong and vargottama—but at only **0°11′ Capricorn**. Braha treats the first degree as severely weak. Conjunction with Jupiter offers benefic support; Mars’s aspect afflicts it. Strong dignity therefore coexists with serious weakness. |
| **Rahu** | In Leo in the 2nd, an unfavourable sign and house according to Braha; at **0°47′**, also subject to the boundary-degree weakness. Financial and family stability require caution in interpretation. No close planetary conjunction or full received planetary aspect is evident. |
| **Ketu** | In the difficult 8th at **0°47′ Aquarius**, without a supplied full benefic aspect. Its dispositor Saturn has strong dignities but major qualifications. Aries Navamsha supplies no special dignity established by the supplied texts. |

**Sources:** Braha’s individual planetary chapters, house-placement chapters, “Retrogrades,” “Planetary Zones” and “Dik Bala”; Harihar, Chapter IV, rules on planetary weakness and mixed house influences.

### Most supported and most pressured areas

- **Career and practical capability:** supported by Mars and its own-house aspect.
- **Gains:** receive aspects from Mars and Jupiter, but their lord Venus is debilitated.
- **Home, education and contentment:** heavily activated but pressured by Mars, afflicted Mercury and Saturn’s aspect.
- **Partnerships:** prominent and potentially consequential, but not straightforwardly peaceful.
- **Personal well-being and family finances:** challenged by the ascendant lord in the 6th and Rahu in the 2nd.

## 4. Important yogas, cancellations and exceptions

### Established, but qualified

- **Neechabhanga for Jupiter:** Capricorn’s lord Saturn is in a kendra and its own sign; Mars, the planet exalted in Capricorn, is also angular.
- **Neechabhanga for Venus:** its dispositor Mercury is angular. Mercury is also the planet exalted in Virgo.

  **Source distinction:** Harihar, Chapter IV, describes applicable cancellation conditions as restoring strength. Braha, “Neechabhanga Yoga,” explicitly warns that the original harm can remain alongside good results. **Mitigation, not complete erasure, is the sound synthesis.**

- **Sasa yoga:** Saturn occupies its own sign in a kendra. The formal condition is satisfied, but the near-zero degree and Mars affliction substantially qualify its effectiveness.  
  **Source:** Harihar, Chapter XIV, yoga 53(e).

- **Sunapha yoga:** Jupiter and Saturn occupy the 2nd from the Moon. This supports capacity for standing and material accomplishment, subject to the planets’ weaknesses.  
  **Sources:** Braha, “Yogas,” no. 9; Harihar, Chapter XIV, no. 39.

- **Sakata yoga:** the Moon is 12th from Jupiter. This supplies a contrary indication of instability and uneven fortune. **Gajakeshari yoga is not established**, because Jupiter is not angular from the Moon.  
  **Source:** Braha, “Yogas,” nos. 5–6.

- **Kala Sarpa yoga:** all seven classical planets lie within the Leo-to-Aquarius semicircle bounded by Rahu and Ketu. Braha associates this with hardship, but it should not override the chart’s other configurations.  
  **Source:** Braha, “Yogas,” no. 2.

- **Kujadosha:** Mars in the 4th in Libra meets Braha’s condition; the stated Scorpio exception does not apply. Harihar allows mitigation through strength and powerful benefic influence, so its severity is qualified, not settled.  
  **Sources:** Braha, “Kujadosha”; Harihar, Chapter VII, sections 13–14.

### Important uncertainties

- The **6th lord Jupiter conjoins the 8th lord Saturn**, suggesting Viparita Raja Yoga under Harihar’s general rule. However, his stricter discussion excludes relationships with other house lords; Mars aspects both. Therefore, a **clean, independently powerful Viparita Raja Yoga is not secure**.
- Braha associates the **Mars–Saturn mutual aspect** with technical excellence; Harihar also treats it as weakening both planets. Technical aptitude and strain can coexist.
- The chart labels Jupiter a functional malefic, while both texts give Cancer’s 9th-lord Jupiter favourable status with 6th-house qualifications. Its actual condition warrants a **mixed assessment**.
- No planet is within Braha’s **8° combustion limit** of the Sun; no eligible planetary pair is within the **1° planetary-war limit**.
- Navamsha conjunctions and aspects have not been used: Braha expressly rejects them. Navamsha-house conclusions would additionally require confidence in birth-time accuracy.

## 5. Overall synthesis

The dominant pattern is **constructive ability operating through difficult conditions**. Mars and the Mercury–Venus exchange provide practical resources; Navamsha adds resilience. Against these stand the weakened ascendant lord, near-exact Jupiter debility, boundary-degree Saturn and nodes, and pronounced domestic and partnership afflictions.

The cancellations are important enough to reject an excessively pessimistic reading, but **not strong enough to justify ignoring the original weaknesses**. This chart’s clearest distinction is between the capacity to accomplish things and the ease with which those accomplishments translate into peace and satisfaction.

## Add Gochar and Transit Info

In [17]:
Gochar = readText(GOCHAR)
Transits = readText(GOCHAR_EVENTS)

/content/Gochar_Chart.txt | 4,584 characters
/content/Transit.txt | 5,494 characters


In [18]:
CONTEXT = augContext(
    BaseCONTEXT,
    FIRSTPASS=FirstPass,
    GOCHAR=Gochar,
    TRANSITS=Transits
)

## Konsultation

In [19]:
# ------------------------------------------------
# Ask your question / define the task
# This and subsequent cells can be run multiple times
# ------------------------------------------------

CurrentQuestion = """
Prepare a chronological 30-day Gochar forecast using the supplied
Gochar, Transit, Natal Chart, Dasha and FirstPass information.

METHODOLOGY

1. INITIAL STATE
   Begin with the supplied Gochar as the baseline planetary configuration.
   Summarise its principal implications before considering future transits.

2. TRANSIT SEQUENCE
   Identify all planetary transits occurring within the next 30 days,
   strictly using the supplied Transit data.

   Process these events chronologically. After each transit, update the
   prevailing Gochar configuration. The updated configuration becomes
   the baseline for analysing the next transit.

   Do not analyse transits as isolated events. Account for the cumulative
   effects of planetary movements.

3. INTERPRETATION
   For the initial Gochar and after each subsequent transit, analyse
   the implications for:

   (a) Career, profession and work
   (b) Family and personal relationships
   (c) Health and physical well-being

   Interpret each configuration in the context of the natal horoscope,
   relevant Dasha/Antardasha periods and established FirstPass findings.

4. PRESENTATION
   For each Gochar configuration:

   - State the effective date and the planetary changes involved.
   - Begin with a clear, substantive explanation in ordinary language,
     understandable to someone unfamiliar with astrology.
   - Follow with a concise technical astrological justification,
     identifying the relevant planets, houses, aspects and Dasha influences.
   - Cite relevant principles from the supplied BrahaHarihar corpus.
     Do not invent quotations or source references.
   - Distinguish significant developments from minor influences.

5. FINAL SYNTHESIS
   Conclude with an integrated assessment of the entire 30-day period:

   - Most favourable periods and their likely significance.
   - Periods requiring caution and the reasons for concern.
   - Important changes in career, relationships and health.
   - Practical guidance based on the overall astrological assessment.

CONSTRAINTS

Use only the supplied astrological data and interpretive sources.
Do not invent planetary positions, transit dates or Dasha transitions.
Avoid repetitive generic predictions and exaggerated certainty.
Treat health interpretations as astrological indications, not medical diagnoses.
Ensure that each successive Gochar reflects all preceding transits.
"""

In [20]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

#model = "gpt-6.1-sol"

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6.1-sol | Effort: medium | In: 228,744 | Cached: 1,335 | Out: 4,751 | Reasoning: 1,034 | Time: 115.2s


In [21]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, Native_id, CurrentQuestion, provider,model)

Saved: /content/K21_261115_105406_gpt-6.1-sol.md
Saved: /content/K21_261115_105406_gpt-6.1-sol.docx


## Check Credit Status
https://platform.openai.com/settings/organization/billing/credit-grants

https://platform.openai.com/usage

https://platform.claude.com/settings/billing

https://platform.claude.com/usage


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)